# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shweta1805/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [11]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", bool(HF_TOKEN))

Token loaded: True


In [12]:
%pip -q install duckdb huggingface_hub

In [13]:
import os
import getpass

HF_TOKEN = os.environ.get('HF_TOKEN')

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    'Paste your Hugging Face READ token (hf_...): '
)

In [14]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("DuckDB connected to FlyRank warehouse.")

DuckDB connected to FlyRank warehouse.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of analysis and time window

One row represents one content item for one client over a 30-day observation window.

The analysis uses daily content-performance data aggregated to the content-item level. The client identifier is retained only for grouping and validation, and no client-identifying information will be published.

A 30-day window provides enough observations to measure recent search performance while keeping the features relevant to the decision.

In [15]:
# Verify the grain: one row per content item and client
query = f"""
SELECT
    client_hash_id,
    content_hash_id,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date,
    COUNT(*) AS daily_rows
FROM {TABLES["fact_daily"]}
WHERE report_date >= DATE '2026-06-01'
  AND report_date < DATE '2026-07-01'
GROUP BY client_hash_id, content_hash_id
ORDER BY daily_rows DESC
LIMIT 10
"""

grain_check = con.sql(query).df()

print("Rows returned:", len(grain_check))
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows returned: 10


,client_hash_id,content_hash_id,start_date,end_date,daily_rows
0,client_e00b29e582949543,content_762f369b413f1869,2026-06-13,2026-06-30,36
1,client_e00b29e582949543,content_81c1fc0cd0d023b9,2026-06-13,2026-06-30,36
2,client_e00b29e582949543,content_8e51f9165149270e,2026-06-13,2026-06-30,36
3,client_e00b29e582949543,content_ac3831ae8497a17e,2026-06-13,2026-06-30,36
4,client_e00b29e582949543,content_734ed63b762224dc,2026-06-13,2026-06-30,36
5,client_e00b29e582949543,content_4b1e89b995d62ac2,2026-06-13,2026-06-30,36
6,client_e00b29e582949543,content_7f26a4c4fdbf62b0,2026-06-13,2026-06-30,36
7,client_e00b29e582949543,content_c58c118161c27095,2026-06-13,2026-06-30,36
8,client_e00b29e582949543,content_4b68573274f30d56,2026-06-13,2026-06-30,36
9,client_810019792c9b8efc,content_00e045559f33deac,2026-06-13,2026-06-30,36


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Fields: feature / label / context / excluded

**Features:** previous-period GSC impressions, GSC clicks, GSC average position, GA4 sessions, and engaged sessions. These describe observed search and engagement performance available before the prediction window.

**Label:** whether GSC impressions decline by more than 20% in the future period. This is a defined outcome calculated from observed impressions.

**Context:** client_hash_id, content_hash_id, report_date, month, and data-availability fields. These help identify, group, filter, and validate observations.

**Excluded:** future-period performance values and any label-derived columns are excluded from the model features because they would leak information from the outcome into the prediction. Client identifiers are also not used as predictive features.

In [16]:
# Show the fields available in the daily performance table

columns = con.sql(f"""
    SELECT column_name, column_type
    FROM (
        DESCRIBE SELECT *
        FROM {TABLES["fact_daily"]}
    )
""").df()

columns

,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [17]:
# Query 1: Verify the grain of the daily table

grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS rows_per_key
    FROM {TABLES["fact_daily"]}
    WHERE month = '2026-03'
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()

print("Duplicate date-client-content combinations:", len(grain_check))
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate date-client-content combinations: 0


,report_date,client_hash_id,content_hash_id,rows_per_key


In [18]:
# Query 2: Check row count and date range for the chosen month

coverage = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {TABLES["fact_daily"]}
    WHERE month = '2026-03'
""").df()

coverage

,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [19]:
# Query 3: Check GSC data availability

availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS FALSE) AS gsc_unavailable_rows
    FROM {TABLES["fact_daily"]}
    WHERE month = '2026-03'
""").df()

availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,gsc_unavailable_rows
0,9841378,3611061,6230317


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Five candidate features

I would use five historical performance features:

1. **Previous-period GSC impressions** — measures the recent search visibility of the content.
2. **Previous-period GSC clicks** — measures how much organic traffic the content received from search.
3. **Previous-period GSC average position** — represents the content's average search ranking.
4. **Previous-period GA4 sessions** — measures recent website traffic to the content.
5. **Previous-period engaged sessions** — measures the quality of recent user engagement.

All five features are calculated from information available before the prediction window. They are therefore intended to provide decision-support without using future outcome information.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.